# Local GraphRAG agent

This notebook builds a knowledge graph from a synthetic travel-agency document collection and compares three ways of answering questions about it:

1. **Classic RAG** — one BM25 retrieval plus one LLM call.
2. **Text-only agent** — a ReAct agent that can call `text_search` (BM25) several times.
3. **Text + graph agent** — the same agent with an extra `graph_search` tool over the extracted knowledge graph.

The collection has 47 documents (hotels, guides, excursions, tours, and agency rules) and a 24-question evaluation set. Multi-hop questions need facts that live in different documents (for example: excursion → guide → languages).

All facts in the dataset are invented, so the model cannot know them in advance.

**How to run.** Install the packages from `requirements.txt`, start [Ollama](https://ollama.com), pull `qwen3:8b`, and optionally create a 16k-context model named `qwen3-hw` (this is what the saved outputs used). Then run all cells from top to bottom. See the README for exact commands. First-time graph extraction makes 47 LLM calls and caches them in `graph_extraction.json`.

**About the saved outputs.** The results already in this notebook come from a local run with Ollama (`qwen3-hw`, i.e. `qwen3:8b` with `num_ctx=16384`) via the OpenAI-compatible API at `http://localhost:11434/v1`. No API key is required.


## 0. Setup

Install the Python dependencies, then connect to a local Ollama model. The placeholder API key `"ollama"` is only there because the OpenAI-compatible client expects a string; it is not a real secret.


In [1]:
%pip install -q "langchain>=1.4,<2" "langchain-openai>=1.7,<2" networkx rank_bm25


Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import os
import re
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from typing import Any, Callable, Optional

import networkx as nx
import numpy as np
from langchain.agents import create_agent
from langchain.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain.tools import BaseTool, ToolException
from langchain_core.tools import StructuredTool
from langchain_openai import ChatOpenAI
from langgraph.errors import GraphRecursionError
from pydantic import BaseModel, Field
from rank_bm25 import BM25Okapi

DATA_PATH = Path("travel_agency_data.json")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Put {DATA_PATH.name} next to this notebook.")

# Local model via Ollama (OpenAI-compatible API on localhost; no real API key needed).
# qwen3-hw is qwen3:8b with a 16k context (ollama create qwen3-hw, num_ctx 16384).
# Override with LLM_BASE_URL / LLM_MODEL if you use a different local model.
LLM_BASE_URL = os.getenv("LLM_BASE_URL", "http://localhost:11434/v1")
LLM_MODEL = os.getenv("LLM_MODEL", "qwen3-hw")
os.environ.setdefault("LLM_API_KEY", "ollama")

llm = ChatOpenAI(
    model=LLM_MODEL,
    base_url=LLM_BASE_URL,
    api_key=os.environ["LLM_API_KEY"],
    temperature=0,
    timeout=600,
    max_retries=2,
    extra_body={"reasoning_effort": "none"},  # disable qwen3 "thinking": faster, answer goes to content
)
print("model:", LLM_MODEL)
print("base_url:", LLM_BASE_URL)
print(llm.invoke("Ответь одним словом: готово").content)


готово


### Data

`travel_agency_data.json` contains:

* `documents` — 47 documents: hotel, guide, excursion and tour cards, plus agency rules. Fields: `doc_id`, `title`, `kind`, `text`.
* `schema` — the graph schema: 6 entity types and 7 relation types, including which entity types may be linked.
* `eval` — 24 questions. `hops` is how many relation steps the answer needs (0 = not in the database); `gold_doc_ids` and `key_facts` describe what a good answer should contain.

The helper functions below are used as-is. Graph extraction is one model call per document (47 calls). The system comparison later adds on the order of 80 more calls. `parallel_map` sends them 8 at a time.


In [3]:
DATA = json.loads(Path("travel_agency_data.json").read_text(encoding="utf-8"))
DOCS: list[dict] = DATA["documents"]
SCHEMA: dict = DATA["schema"]
EVAL: list[dict] = DATA["eval"]
DOC_BY_ID = {doc["doc_id"]: doc for doc in DOCS}

MAX_WORKERS = 1  # локальная модель на одной видеокарте обрабатывает запросы по очереди

def parallel_map(fn: Callable, items: list, desc: str = "") -> list:
    """fn(item) для всех items в несколько потоков; результаты в исходном порядке."""
    results, errors, done = [None] * len(items), [], 0
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {pool.submit(fn, item): index for index, item in enumerate(items)}
        for future in as_completed(futures):
            index = futures[future]
            try:
                results[index] = future.result()
            except Exception as error: 
                errors.append((index, error))
            done += 1
            if done % max(1, len(items) // 5) == 0 or done == len(items):
                print(f"{desc}: готово {done}/{len(items)}", flush=True)
    if errors:
        index, error = min(errors, key=lambda pair: pair[0])
        raise RuntimeError(f"{len(errors)} из {len(items)} задач упали; первая — элемент {index}: {error!r}") from error
    return results

def tokenize(text: str) -> list[str]:
    return re.findall(r"[a-zа-яё0-9]+", text.lower())

def extract_citations(answer: str) -> list[str]:
    """doc_id из квадратных скобок в ответе, уникальные, по порядку."""
    found: list[str] = []
    for group in re.findall(r"\[([^\]]+)\]", answer or ""):
        for token in re.split(r"[,;\s]+", group):
            if re.fullmatch(r"[a-z]+_[a-z0-9_]+", token) and token not in found:
                found.append(token)
    return found

def has_no_info(answer: str) -> bool:
    return bool(re.search(r"нет информаци|нет данных|не нашёл|не нашел|не найден|нет в базе|не располага|отсутству|такого .* нет",
                          answer or "", re.IGNORECASE))

print("Типы сущностей:", ", ".join(SCHEMA["entity_types"]))
for name, rule in SCHEMA["relations"].items():
    print(f"  {name}: {rule['source']} → {rule['target']} — {rule['meaning']}")
print("\nДокументов:", len(DOCS), "| вопросов:", len(EVAL))
print("\nПример:", DOC_BY_ID["e_kzn_food"]["text"])


Типы сущностей: Город, Отель, Экскурсия, Гид, Язык, Тур
  находится_в: Отель|Экскурсия → Город — отель стоит в городе / экскурсия проходит в городе
  работает_в: Гид → Город — город, где работает гид
  ведёт: Гид → Экскурсия — гид проводит экскурсию
  говорит_на: Гид → Язык — гид может вести экскурсии на этом языке
  стартует_от: Экскурсия → Отель — сбор группы у отеля или в его холле
  включает: Тур → Экскурсия — экскурсия входит в программу тура
  проживание_в: Тур → Отель — участники тура живут в этом отеле

Документов: 47 | вопросов: 24

Пример: «Гастротур по Казани» — авторская экскурсия в Казани: чак-чак, эчпочмак и губадия. Длительность 3 ч, стоимость 2900 рублей с человека.

Встреча у входа на Колхозный рынок. Ведёт экскурсию Алия.


## 1. Чтение документов моделью

В этой части граф ещё не строится. Модель читает каждый документ отдельно и перечисляет, какие сущности в нём упомянуты и как они связаны, по схеме `SCHEMA`. Для этого нужно:

1. написать промпт;
2. проверить правильность данных, извлечённых моделью.

На выходе должно получиться 47 проверенных списков сущностей и связей, по одному на документ. В части 2 вы соберёте из них граф.

### 1.1. Промпт извлечения

Граф строится из текста: модель читает документ и перечисляет, какие сущности в нём упомянуты и как они связаны.

Например, из карточки экскурсии `e_kzn_food` («Гастротур по Казани» — авторская экскурсия в Казани … Ведёт экскурсию Алия.) должно получиться примерно такое:

```json
{"entities": [{"name": "Гастротур по Казани", "type": "Экскурсия"},
              {"name": "Казань", "type": "Город"},
              {"name": "Алия", "type": "Гид"}],
 "relations": [{"source": "Гастротур по Казани", "relation": "находится_в", "target": "Казань",
                "evidence": "авторская экскурсия в Казани"},
               {"source": "Алия", "relation": "ведёт", "target": "Гастротур по Казани",
                "evidence": "Ведёт экскурсию Алия"}]}
```

От этого промпта зависит качество всего графа: то, что модель здесь пропустит, не попадёт в граф, а то, что придумает, окажется в нём лишним.

Напишите `build_extraction_prompt(doc, schema) -> list[dict]` — сообщения для `llm.invoke` в формате `[{"role": "system", "content": ...}, {"role": "user", "content": ...}]`. В промпте должны быть:

* все типы сущностей и связей схемы с пояснениями, для связей — между какими типами она допустима;
* правила записи имён: как писать город (именительный падеж), язык, нужно ли убирать кавычки и слова «отель», «гостиница»;
* просьба извлекать только то, что сказано в документе;
* ответ — только JSON вида `{"entities": [{"name", "type"}], "relations": [{"source", "relation", "target", "evidence"}]}`, где `evidence` — точная цитата из документа, подтверждающая связь;
* текст документа целиком.


In [4]:
def build_extraction_prompt(doc: dict, schema: dict) -> list[dict]:
    entity_lines = "\n".join(f"- {etype}: {meaning}" for etype, meaning in schema["entity_types"].items())
    relation_lines = "\n".join(
        f"- {name}: {rule['source'].replace('|', ' или ')} → {rule['target'].replace('|', ' или ')} — {rule['meaning']}"
        for name, rule in schema["relations"].items()
    )
    allowed_types = ", ".join(f'"{t}"' for t in schema["entity_types"])
    allowed_relations = ", ".join(f'"{r}"' for r in schema["relations"])
    example = {
        "entities": [{"name": "Гастротур по Казани", "type": "Экскурсия"},
                     {"name": "Казань", "type": "Город"},
                     {"name": "Алия", "type": "Гид"}],
        "relations": [{"source": "Гастротур по Казани", "relation": "находится_в", "target": "Казань",
                       "evidence": "авторская экскурсия в Казани"},
                      {"source": "Алия", "relation": "ведёт", "target": "Гастротур по Казани",
                       "evidence": "Ведёт экскурсию Алия"}],
    }

    system = f"""Ты извлекаешь граф знаний из одного документа турагентства «Попутный ветер».
Твоя задача — перечислить сущности, упомянутые в документе, и связи между ними строго по схеме ниже.

ТИПЫ СУЩНОСТЕЙ (других типов не бывает):
{entity_lines}

ТИПЫ СВЯЗЕЙ (направление: источник → цель; других связей не бывает):
{relation_lines}

ПРАВИЛА ИЗВЛЕЧЕНИЯ
1. Извлекай только то, что прямо сказано в тексте документа. Ничего не додумывай и не бери из своих знаний:
   если гид назван только по имени («Алия») или только по фамилии («Соколова») — так и пиши, не добавляй вторую часть имени.
2. Объекты, которые не подходят ни под один тип (рынок, станция метро, улица, озеро, гора, мечеть, рестораны, агентство, правила, цены),
   НЕ извлекай. Цены, длительность и прочие атрибуты не нужны — только сущности и связи.
3. Связь добавляй, только если у неё есть подходящий тип и типы концов допустимы (например, «ведёт» — всегда от Гида к Экскурсии,
   а не наоборот; «включает» — от Тура к Экскурсии; «проживание_в» — от Тура к Отелю; «стартует_от» — от Экскурсии к Отелю,
   только если сбор/встреча у отеля или в его холле; встреча у рынка, станции, студии — это не «стартует_от»).
4. Каждый конец связи (source и target) должен буквально совпадать с полем name одной из сущностей в списке entities.
5. Если в документе нет ни одной сущности или связи по схеме (например, это правило агентства), верни пустые списки.

ПРАВИЛА ЗАПИСИ ИМЁН (name)
- Пиши имя без кавычек «» "" и без родовых слов «отель», «гостиница», «экскурсия», «тур», «гид», «город»:
  «Гостиница «Кабан»» → «Кабан»; «Отель «Петроградская сторона»» → «Петроградская сторона»; «экскурсия «Гастротур по Казани»» → «Гастротур по Казани».
- Короткое название отеля пиши так, как оно встретилось («Петроградка»); не подменяй его другим названием.
- Город — в именительном падеже, полное официальное название: «в Казани» → «Казань», «в Санкт-Петербурге» → «Санкт-Петербург»,
  «в Москве» → «Москва», «в Ереване» → «Ереван», «в Стамбуле» → «Стамбул».
- Язык — одно слово в именительном падеже мужского рода, со строчной буквы, без слова «язык»:
  «на русском, татарском и английском языках» → «русский», «татарский», «английский»; «по-немецки» → «немецкий».
- Людей пиши в именительном падеже, как в документе: «Алия Гарипова», «Мерве», «Сафин».
- Названия экскурсий и туров — полностью, как в документе, с сохранением регистра, но без кавычек.
- Одна сущность — одна запись в entities, без повторов.

ПОЛЕ evidence
- evidence — ТОЧНАЯ непрерывная цитата из текста документа (не из заголовка), которая подтверждает именно эту связь.
- Копируй символ в символ: те же слова, падежи, буквы (ё), кавычки и знаки. Не пересказывай, не сокращай многоточием,
  не склеивай куски из разных мест. Достаточно короткого фрагмента (3–15 слов) или одного предложения.
- Одна цитата может подтверждать несколько связей (например, перечисление языков гида).

ФОРМАТ ОТВЕТА
Ответ — только JSON-объект, без пояснений, без markdown и без блока ```:
{{"entities": [{{"name": "...", "type": "..."}}], "relations": [{{"source": "...", "relation": "...", "target": "...", "evidence": "..."}}]}}
Допустимые значения type: {allowed_types}.
Допустимые значения relation: {allowed_relations}.

ПРИМЕР. Документ: «Гастротур по Казани» — авторская экскурсия в Казани … Встреча у входа на Колхозный рынок. Ведёт экскурсию Алия.
Ответ: {json.dumps(example, ensure_ascii=False)}
(Колхозный рынок — не отель, поэтому связи «стартует_от» нет.)"""

    user = (f"doc_id: {doc.get('doc_id', '')}\n"
            f"Заголовок (только для понимания, цитаты из него не брать): {doc.get('title', '')}\n\n"
            f"Текст документа:\n<<<\n{doc['text']}\n>>>\n\n"
            "Извлеки сущности и связи по схеме и верни только JSON.")
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]



### 1.2. Разбор и проверка ответа

Ответ модели на промпт из 1.1 — это текст, внутри которого, если всё прошло хорошо, лежит JSON со списками сущностей и связей. Чтобы собрать из него граф, этот JSON нужно достать из текста и проверить. Всё, что пройдёт проверку, попадёт в итоговый граф.

Модель может ошибаться: оборачивает JSON в текст, придумывает типы, путает направление связей, приводит «цитаты», которых нет в документе.

Напишите `parse_extraction(text, doc_text, schema) -> dict | None`, который:

* находит JSON в ответе (в том числе внутри текста или блока кода); если JSON нет или в нём нет списков `entities` и `relations` — `None`; функция не падает ни на каком ответе;
* оставляет сущности только с типами из схемы;
* оставляет связь, только если: связь есть в схеме; оба конца есть среди сущностей этого ответа; типы концов допустимы для этой связи (`"Отель|Экскурсия"` — любой из двух); `evidence` действительно встречается в тексте документа (без учёта регистра, лишних пробелов и разницы ё/е);
* возвращает `{"entities": [...], "relations": [...], "dropped": [...]}`, где `dropped` — отброшенные связи с полем `reason`: почему отброшена. Это понадобится в 1.3.


In [5]:
def parse_extraction(text: str, doc_text: str, schema: dict) -> Optional[dict]:
    def norm_text(value: Any) -> str:
        value = str(value).lower().replace("ё", "е")
        value = re.sub(r"[«»“”„\"'`]", "", value)            # кавычки модель часто меняет
        value = value.replace("—", "-").replace("–", "-")
        return re.sub(r"\s+", " ", value).strip()

    def norm_key(value: Any) -> str:
        return re.sub(r"[\s\-]+", "_", str(value).strip().lower().replace("ё", "е"))

    def find_json(raw: str) -> Optional[dict]:
        candidates = re.findall(r"```(?:json|JSON)?\s*(.*?)```", raw, re.DOTALL) + [raw]
        decoder = json.JSONDecoder()
        for chunk in candidates:
            chunk = chunk.strip()
            for start in [m.start() for m in re.finditer(r"\{", chunk)]:
                try:
                    obj, _ = decoder.raw_decode(chunk[start:])
                except (json.JSONDecodeError, ValueError, RecursionError):
                    continue
                if isinstance(obj, dict) and isinstance(obj.get("entities"), list) and isinstance(obj.get("relations"), list):
                    return obj
        return None

    try:
        if not isinstance(text, str) or not text.strip():
            return None
        data = find_json(text)
        if data is None:
            return None

        type_by_key = {norm_key(t): t for t in schema["entity_types"]}
        rel_by_key = {norm_key(r): r for r in schema["relations"]}
        doc_norm = norm_text(doc_text or "")

        entities, dropped_entities = [], []
        types_by_name: dict[str, set] = defaultdict(set)   # нормализованное имя -> типы
        canon_name: dict[str, str] = {}                     # нормализованное имя -> имя как в entities
        seen = set()
        for ent in data["entities"]:
            if not isinstance(ent, dict) or not isinstance(ent.get("name"), str) or not ent["name"].strip():
                dropped_entities.append({"entity": ent, "reason": "сущность без имени"})
                continue
            etype = type_by_key.get(norm_key(ent.get("type", "")))
            name = re.sub(r"\s+", " ", ent["name"]).strip()
            if etype is None:
                dropped_entities.append({"name": name, "type": ent.get("type"), "reason": "тип не из схемы"})
                continue
            key = norm_text(name)
            if (key, etype) in seen:
                continue
            seen.add((key, etype))
            entities.append({"name": name, "type": etype})
            types_by_name[key].add(etype)
            canon_name.setdefault(key, name)

        relations, dropped = [], []
        seen_rel = set()
        for rel in data["relations"]:
            if not isinstance(rel, dict):
                dropped.append({"relation": None, "reason": "связь не словарь", "evidence": str(rel)})
                continue
            item = {k: rel.get(k) for k in ("source", "relation", "target", "evidence")}
            relation = rel_by_key.get(norm_key(item["relation"] or ""))
            if relation is None:
                dropped.append({**item, "reason": "связь не из схемы"})
                continue
            src_key = norm_text(item["source"] or "")
            tgt_key = norm_text(item["target"] or "")
            if src_key not in types_by_name or tgt_key not in types_by_name:
                dropped.append({**item, "reason": "конец связи не среди сущностей"})
                continue
            rule = schema["relations"][relation]
            ok_src, ok_tgt = set(rule["source"].split("|")), set(rule["target"].split("|"))
            src_types, tgt_types = types_by_name[src_key], types_by_name[tgt_key]
            if not (src_types & ok_src and tgt_types & ok_tgt):
                reversed_fits = bool(tgt_types & ok_src and src_types & ok_tgt)
                dropped.append({**item, "reason": "перепутано направление" if reversed_fits else "неподходящие типы концов"})
                continue
            evidence = item["evidence"]
            if not isinstance(evidence, str) or not norm_text(evidence) or norm_text(evidence) not in doc_norm:
                dropped.append({**item, "reason": "цитаты нет в документе"})
                continue
            triple = (src_key, relation, tgt_key)
            if triple in seen_rel:
                continue
            seen_rel.add(triple)
            relations.append({"source": canon_name[src_key], "relation": relation, "target": canon_name[tgt_key],
                              "evidence": evidence.strip()})

        return {"entities": entities, "relations": relations, "dropped": dropped, "dropped_entities": dropped_entities}
    except Exception:  # разбор ответа модели никогда не должен ронять ноутбук
        return None



Запустим модель на всех документах (47 запросов к модели, по 8 одновременно).

Результат сохраняется в `graph_extraction.json`: при перезапуске ноутбука запросы не повторяются. Чтобы извлечь заново (например, после правки промпта), поставьте `REEXTRACT = True`.


In [6]:
REEXTRACT = False
CACHE = Path("graph_extraction.json")

def extract_one(doc: dict) -> dict:
    reply = llm.invoke(build_extraction_prompt(doc, SCHEMA)).text
    parsed = parse_extraction(reply, doc["text"], SCHEMA)
    return {"doc_id": doc["doc_id"], "raw": reply, "parsed": parsed}

if CACHE.exists() and not REEXTRACT:
    EXTRACTIONS = json.loads(CACHE.read_text(encoding="utf-8"))
    print("Загружено из", CACHE)
else:
    EXTRACTIONS = parallel_map(extract_one, DOCS, "извлечение")
    CACHE.write_text(json.dumps(EXTRACTIONS, ensure_ascii=False, indent=1), encoding="utf-8")

failed = [e["doc_id"] for e in EXTRACTIONS if e["parsed"] is None]
parsed = [e for e in EXTRACTIONS if e["parsed"] is not None]
entity_types = Counter(ent["type"] for e in parsed for ent in e["parsed"]["entities"])
relation_types = Counter(rel["relation"] for e in parsed for rel in e["parsed"]["relations"])
dropped = [(e["doc_id"], d) for e in parsed for d in e["parsed"]["dropped"]]

print("Ответ не разобрался:", failed or "ни разу")
print("Сущностей по типам:", dict(entity_types))
print("Связей по типам:", dict(relation_types))
print(f"Отброшено связей: {len(dropped)}", dict(Counter(d['reason'] for _, d in dropped)))
for doc_id, d in dropped[:12]:
    print(f"  [{doc_id}] {d.get('source')} —{d.get('relation')}→ {d.get('target')} | {d['reason']} | «{str(d.get('evidence'))[:60]}»")


Загружено из graph_extraction.json
Ответ не разобрался: ни разу
Сущностей по типам: {'Отель': 31, 'Город': 44, 'Гид': 27, 'Язык': 4, 'Экскурсия': 26, 'Тур': 5}
Связей по типам: {'находится_в': 30, 'работает_в': 16, 'говорит_на': 4, 'стартует_от': 12, 'ведёт': 15, 'проживание_в': 7, 'включает': 11}
Отброшено связей: 26 {'неподходящие типы концов': 1, 'конец связи не среди сущностей': 24, 'перепутано направление': 1}
  [h_zamosk] Замоскворечье —проживание_в→ Москва | неподходящие типы концов | «В Москве мы часто селим туристов в «Замоскворечье»»
  [g_safin] Марат Сафин —говорит_на→ русский | конец связи не среди сущностей | «Языки: русский, немецкий»
  [g_safin] Марат Сафин —говорит_на→ немецкий | конец связи не среди сущностей | «Языки: русский, немецкий»
  [g_beridze] Нино Беридзе —говорит_на→ грузинский | конец связи не среди сущностей | «ведёт экскурсии на грузинском, русском и английском языках»
  [g_beridze] Нино Беридзе —говорит_на→ русский | конец связи не среди сущностей | «ведё

### 1.3. Что извлекла модель

По выводу выше ответьте (4–6 предложений), ссылаясь на `doc_id`:

1. Каких связей и сущностей получилось меньше или больше, чем вы ожидали, глядя на документы?
2. Какие связи отброшены и почему — это ошибки модели или ваша проверка слишком строгая? Приведите два-три примера.
3. Что вы поменяли в промпте 1.1 после первого извлечения и что от этого изменилось? Если не меняли — что поменяли бы.

**Analysis / Разбор:**  Связи «стартует_от» (12), «ведёт» (15), «проживание_в» (7) и «включает» (11) извлеклись ровно в том количестве, которое видно по документам. Сильно меньше ожидаемого получилось языков: всего 4 сущности «Язык» и 4 связи «говорит_на», хотя в карточках гидов их около 25. Наоборот, «работает_в» получилось 16 вместо 11 и «находится_в» 30 вместо 27: модель добавляла лишнее, например «Алия Гарипова —работает_в→ Казань» по цитате из [e_kzn_sloboda] «Сбор группы в холле отеля «Кабан»», которая эту связь не подтверждает, и «Гастротур по Казани —находится_в→ Казань» из [t_volga]. Из 26 отброшенных связей 24 отброшены с причиной «конец связи не среди сущностей»: в [g_safin], [g_beridze], [g_kiknadze] модель написала связь «говорит_на», но не добавила языки в entities, а в [g_mkrtchyan] ещё и оставила их в предложном падеже («армянском»). Это ошибка модели, но проверка здесь, пожалуй, слишком строгая: у «говорит_на» цель может быть только языком, и такую сущность можно было бы добавить автоматически. Ещё одна связь отброшена справедливо: [h_zamosk] «Замоскворечье —проживание_в→ Москва» вместо «находится_в». Промпт после первого извлечения я не менял; поменял бы так: отдельно потребовал бы, чтобы каждый конец связи был в entities, и добавил бы пример с карточкой гида.


## 2. Собираем граф из ответов модели

После части 1 у вас 47 отдельных списков сущностей и связей между ними, по одному списку на документ. Чтобы ответить на вопрос пользователя, их нужно соединить в один граф. Тогда от экскурсии можно перейти по ребру «ведёт» к гиду, а от гида по рёбрам «говорит_на» — к языкам, хотя эти факты записаны в разных документах.

Этим займётся класс `KnowledgeGraph`: он собирает граф в `networkx.MultiDiGraph` (`self.graph`) и ищет по нему. Обход соседей и путей даёт `networkx`, вы пишете логику предметной области. В части 3 по этому классу вы сделаете инструмент поиска для агента.

* Вершина — сущность. Атрибуты: `name` (самое полное из встреченных имён), `type`, `aliases` (множество нормализованных имён), `doc_ids` (множество документов, где она встретилась). Идентификатор вершины — на ваше усмотрение.
* Ребро — связь. Атрибуты: `relation`, `doc_id` (откуда извлечена), `evidence` (цитата).

Методы:

* **2.1 `resolve(name, etype) -> node | None` (1).** Ищет среди вершин того же типа ту, к которой относится упоминание. Зачем: одна сущность в разных документах названа по-разному, и без слияния в графе окажутся две вершины.

  > Тогда цепочка рвётся: в карточке экскурсии гид — «Алия», а языки записаны у вершины «Алия Гарипова», и от экскурсии до языков не дойти. Примеры разных написаний: «Казань» и «казань», «Кабан» и «Гостиница „Кабан“», «Алия» и «Алия Гарипова», «Петроградка» и «Отель „Петроградская сторона“».

  Правила сопоставления придумайте сами: нормализация, совпадение по словам, по началу слова, `difflib` — что сочтёте разумным. Если подходящих вершин несколько или нет ни одной — `None`. Слияние не должно зависеть от порядка документов.
* **2.2 `add_extraction(doc_id, extraction)` (0.25).** Добавляет в граф `self.graph` сущности и связи одного документа — то, что для него вернула `parse_extraction`: каждую сущность — через `resolve` (новая вершина или существующая), каждую связь — ребром между вершинами её концов. Повторное добавление того же документа не создаёт дублей.
* **2.3 `find(query, etype=None, limit=5) -> list[node]` (0.25).** Ищет в графе вершины, которые упомянуты в запросе, и возвращает их идентификаторы: запрос может быть названием («гастротур») или вопросом целиком. Самые подходящие — первыми.
* **2.4 `expand(nodes, depth=2, relations=None, max_edges=30) -> list[dict]` (1).** Собирает подграф — часть графа вокруг заданных вершин, по которой можно пройти цепочку. В него входят рёбра на расстоянии до `depth` шагов, в обе стороны. `relations` ограничивает, по каким связям идти. Не больше `max_edges` рёбер, без повторов. Ребро — словарь `source`, `relation`, `target`, `doc_id`, `evidence`.
* **2.5 `evidence(edges) -> str` (0.25).** Превращает список рёбер из `expand` в текст, который инструмент поиска вернёт модели в части 3: по строке на ребро — имена и типы концов, связь, `[doc_id]` и цитата. Для пустого списка — понятная фраза.


In [7]:
class KnowledgeGraph:
    # Родовые слова и служебные слова, которые не отличают одну сущность от другой
    GENERIC = {"отель", "отеля", "отеле", "гостиница", "гостиницы", "гостинице", "экскурсия", "экскурсии", "экскурсию",
               "тур", "тура", "туре", "гид", "гида", "гиды", "гидов", "город", "города", "г", "язык", "языке", "языка",
               "языках", "языки"}
    STOP = {"и", "в", "во", "на", "по", "за", "с", "со", "у", "от", "до", "из", "к", "о", "об", "для", "кто", "что",
            "как", "какой", "какая", "какие", "каких", "каком", "ли", "есть", "сколько", "где", "при"}

    HUB_TYPES = {"Город", "Язык"}

    def __init__(self):
        self.graph = nx.MultiDiGraph()

    # ---------- нормализация ----------
    @staticmethod
    def _norm(name: str) -> str:
        name = str(name).lower().replace("ё", "е")
        name = re.sub(r"[«»“”„\"'`]", " ", name)
        name = re.sub(r"[\-–—]", " ", name)
        name = re.sub(r"\bпо\s+(?=[а-я]+ски\b)", "", name)          # «по-немецки» → «немецки»
        return re.sub(r"\s+", " ", name).strip()

    @classmethod
    def _canon_language(cls, token: str) -> str:
        # русском / русского / русская / по-русски → русский
        token = re.sub(r"(ск|цк)(ом|ого|ому|им|ая|ой|ую|ие|их|и)$", r"\1ий", token)
        return token

    @classmethod
    def _tokens(cls, name: str, etype: Optional[str] = None) -> list[str]:
        tokens = [t for t in re.findall(r"[a-zа-я0-9]+", cls._norm(name)) if t not in cls.GENERIC and t not in cls.STOP]
        if etype == "Язык":
            tokens = [cls._canon_language(t) for t in tokens]
        return tokens

    @classmethod
    def _alias(cls, name: str, etype: Optional[str] = None) -> str:
        tokens = cls._tokens(name, etype)
        return " ".join(tokens) if tokens else cls._norm(name)

    @staticmethod
    def _token_match(a: str, b: str) -> bool:
        """Совпадение слов с учётом русских окончаний: казань/казани, петроградка/петроградская."""
        if a == b:
            return True
        if a.isdigit() or b.isdigit():
            return False
        shorter = min(len(a), len(b))
        if shorter < 4:
            return False
        prefix = len(os.path.commonprefix([a, b]))
        return prefix >= max(4, shorter - 3)

    @classmethod
    def _covered(cls, small: list[str], big: list[str]) -> bool:
        """Каждое слово small совпадает с каким-то словом big."""
        return bool(small) and all(any(cls._token_match(s, b) for b in big) for s in small)

    def _display_name(self, names: set) -> str:
        # самое полное имя: больше значимых слов; при равенстве — короче (обычно именительный падеж) и с заглавной
        def key(n):
            return (-len(self._tokens(n)), not n[:1].isupper(), len(n), n)
        return sorted(names, key=key)[0]

    # ---------- 2.1 ----------
    def resolve(self, name: str, etype: str) -> Optional[str]:
        if not isinstance(name, str) or not name.strip():
            return None
        alias = self._alias(name, etype)
        tokens = alias.split()
        same_type = [(n, d) for n, d in self.graph.nodes(data=True) if d.get("type") == etype]

        exact = [n for n, d in same_type if alias in d["aliases"]]
        if len(exact) == 1:
            return exact[0]
        if len(exact) > 1:
            return None

        candidates = []
        for n, d in same_type:
            for other in d["aliases"]:
                other_tokens = other.split()
                # одно имя — часть другого: «Алия» ⊂ «Алия Гарипова», «Кабан» ⊂ «Гостиница Кабан»,
                # «Петроградка» ~ «Петроградская сторона», «казани» ~ «казань»
                if self._covered(tokens, other_tokens) or self._covered(other_tokens, tokens):
                    candidates.append(n)
                    break
        if len(candidates) == 1:
            return candidates[0]
        return None  # нет ни одной или неоднозначно

    # ---------- 2.2 ----------
    def _new_node_id(self, name: str, etype: str) -> str:
        base = f"{etype}:{self._alias(name, etype)}"
        node_id, i = base, 2
        while node_id in self.graph:
            node_id, i = f"{base}#{i}", i + 1
        return node_id

    def _add_entity(self, doc_id: str, name: str, etype: str) -> str:
        name = re.sub(r"\s+", " ", name).strip()
        if etype == "Язык":  # «русском», «по-русски» → «русский»
            name = self._alias(name, etype) or name
        node = self.resolve(name, etype)
        if node is None:
            node = self._new_node_id(name, etype)
            self.graph.add_node(node, name=name, type=etype, aliases=set(), doc_ids=set(), raw_names=set())
        data = self.graph.nodes[node]
        data["aliases"].add(self._alias(name, etype))
        data["doc_ids"].add(doc_id)
        data["raw_names"].add(name)
        data["name"] = self._display_name(data["raw_names"])
        return node

    def add_extraction(self, doc_id: str, extraction: dict) -> None:
        if not extraction:
            return
        node_by_name: dict[str, list[str]] = defaultdict(list)
        for ent in extraction.get("entities", []):
            name, etype = ent.get("name"), ent.get("type")
            if not isinstance(name, str) or not name.strip() or not etype:
                continue
            node = self._add_entity(doc_id, name, etype)
            node_by_name[self._norm(name)].append(node)

        relations_schema = SCHEMA["relations"] if "SCHEMA" in globals() else {}
        for rel in extraction.get("relations", []):
            relation = rel.get("relation")
            sources = node_by_name.get(self._norm(rel.get("source", "")), [])
            targets = node_by_name.get(self._norm(rel.get("target", "")), [])
            rule = relations_schema.get(relation)
            if rule:  # если имя встречается с разными типами — берём допустимые схемой
                sources = [s for s in sources if self.graph.nodes[s]["type"] in rule["source"].split("|")] or sources
                targets = [t for t in targets if self.graph.nodes[t]["type"] in rule["target"].split("|")] or targets
            if not sources or not targets:
                continue
            u, v = sources[0], targets[0]
            # ключ ребра = связь + документ: повторное добавление документа обновляет ребро, а не дублирует
            self.graph.add_edge(u, v, key=f"{relation}@{doc_id}", relation=relation, doc_id=doc_id,
                                evidence=rel.get("evidence", ""))

    # ---------- 2.3 ----------
    def find(self, query: str, etype: Optional[str] = None, limit: int = 5) -> list[str]:
        if not isinstance(query, str) or not query.strip():
            return []
        q_tokens = self._tokens(query)
        q_lang = [self._canon_language(t) for t in q_tokens]
        q_norm = " " + " ".join(q_tokens) + " "
        scored = []
        for n, d in self.graph.nodes(data=True):
            if etype and d["type"] != etype:
                continue
            best = None
            tokens_query = q_lang if d["type"] == "Язык" else q_tokens
            for alias in d["aliases"]:
                a_tokens = alias.split()
                if not a_tokens:
                    continue
                matched = [a for a in a_tokens if any(self._token_match(a, q) for q in tokens_query)]
                if not matched:
                    continue
                coverage = len(matched) / len(a_tokens)
                if coverage < 0.5:
                    continue
                exact_phrase = f" {alias} " in q_norm
                score = (coverage + (0.5 if exact_phrase else 0), sum(len(m) for m in matched))
                best = score if best is None or score > best else best
            if best is not None:
                scored.append((best, d["name"], n))
        scored.sort(key=lambda x: (-x[0][0], -x[0][1], x[1]))
        return [n for _, _, n in scored[:limit]]

    # ---------- 2.4 ----------
    def _edge_dict(self, u: str, v: str, data: dict) -> dict:
        return {"source": self.graph.nodes[u]["name"], "relation": data["relation"], "target": self.graph.nodes[v]["name"],
                "doc_id": data["doc_id"], "evidence": data.get("evidence", ""),
                "source_type": self.graph.nodes[u]["type"], "target_type": self.graph.nodes[v]["type"],
                "source_id": u, "target_id": v}

    def expand(self, nodes: list[str], depth: int = 2, relations: Optional[list[str]] = None, max_edges: int = 30) -> list[dict]:
        if isinstance(nodes, str):
            nodes = [nodes]
        frontier = [n for n in dict.fromkeys(nodes or []) if n in self.graph]
        allowed = set(relations) if relations else None
        visited = set(frontier)
        seen_edges, result = set(), []
        for _ in range(max(0, int(depth))):
            next_frontier = []
            for node in frontier:
                incident = [(u, v, k, d) for u, v, k, d in self.graph.out_edges(node, keys=True, data=True)]
                incident += [(u, v, k, d) for u, v, k, d in self.graph.in_edges(node, keys=True, data=True)]
                for u, v, k, d in sorted(incident, key=lambda e: (e[3]["relation"], e[0], e[1], str(e[2]))):
                    if allowed is not None and d["relation"] not in allowed:
                        continue
                    if (u, v, k) in seen_edges:
                        continue
                    seen_edges.add((u, v, k))
                    result.append(self._edge_dict(u, v, d))
                    if len(result) >= max_edges:
                        return result
                    other = v if u == node else u
                    # Города и языки — «хабы»: с ними связаны десятки вершин. Ребро до хаба берём, но дальше через него
                    # не идём (иначе подграф тура заполняется всеми отелями города и всеми гидами, говорящими по-русски).
                    # Если хаб — стартовая вершина (вопрос про город или язык), он раскрывается как обычно.
                    if self.graph.nodes[other]["type"] in self.HUB_TYPES:
                        visited.add(other)
                        continue
                    if other not in visited:
                        visited.add(other)
                        next_frontier.append(other)
            frontier = next_frontier
            if not frontier:
                break
        return result

    # ---------- 2.5 ----------
    def evidence(self, edges: list[dict]) -> str:
        if not edges:
            return "В графе не найдено связей для этих сущностей."
        lines = []
        for e in edges:
            src_type = f" ({e['source_type']})" if e.get("source_type") else ""
            tgt_type = f" ({e['target_type']})" if e.get("target_type") else ""
            lines.append(f"{e['source']}{src_type} —{e['relation']}→ {e['target']}{tgt_type} [{e['doc_id']}] «{e.get('evidence', '')}»")
        return "\n".join(lines)


Соберём граф из своих извлечений и посмотрим на него. Список «похожих имён» — кандидаты на ошибки слияния: одна сущность разбита на две вершины или, наоборот, разные сущности склеены.


In [9]:
from difflib import SequenceMatcher

kg = KnowledgeGraph()
for item in EXTRACTIONS:
    if item["parsed"] is not None:
        kg.add_extraction(item["doc_id"], item["parsed"])

nodes = kg.graph.nodes(data=True)
print("Вершин:", kg.graph.number_of_nodes(), "| рёбер:", kg.graph.number_of_edges())
print("Вершины по типам:", dict(Counter(d["type"] for _, d in nodes)))
print("Рёбра по связям:", dict(Counter(d["relation"] for _, _, d in kg.graph.edges(data=True))))
lonely = [d["name"] for n, d in nodes if kg.graph.degree(n) == 0]
print("Вершины без связей:", lonely or "нет")

def _plain(name: str) -> str:
    name = re.sub(r"[«»'\"]", "", name.lower().replace("ё", "е"))
    return re.sub(r"^(отель|гостиница|экскурсия|тур|гид)\s+", "", name).strip()

print("\nПохожие имена одного типа (возможно, одна сущность разбита на две):")
by_type = defaultdict(list)
for n, d in nodes:
    by_type[d["type"]].append(d["name"])
similar = 0
for etype, names in by_type.items():
    for i, a in enumerate(names):
        for b in names[i + 1:]:
            pa, pb = _plain(a), _plain(b)
            if pa in pb or pb in pa or len(os.path.commonprefix([pa, pb])) >= 5 or SequenceMatcher(None, pa, pb).ratio() > 0.8:
                print(f"  {etype}: «{a}» / «{b}»")
                similar += 1
if not similar:
    print("  не найдено")

print("\nСклеенные вершины (несколько разных имён):")
for n, d in nodes:
    if len(d["aliases"]) > 1:
        print(f"  {d['type']}: {d['name']} ← {sorted(d['aliases'])}")

guide = kg.find("Алия Гарипова", etype="Гид")
if guide:
    print("\nПример подграфа:")
    print(kg.evidence(kg.expand(guide, depth=1)))


Вершин: 55 | рёбер: 95
Вершины по типам: {'Отель': 12, 'Город': 7, 'Гид': 12, 'Язык': 4, 'Экскурсия': 15, 'Тур': 5}
Рёбра по связям: {'находится_в': 30, 'работает_в': 16, 'говорит_на': 4, 'ведёт': 15, 'стартует_от': 12, 'проживание_в': 7, 'включает': 11}
Вершины без связей: ['агентство']

Похожие имена одного типа (возможно, одна сущность разбита на две):
  не найдено

Склеенные вершины (несколько разных имён):
  Отель: Петроградская сторона ← ['петроградка', 'петроградская сторона']
  Гид: Алия Гарипова ← ['алия', 'алия гарипова']
  Гид: Марат Сафин ← ['марат сафин', 'сафин']
  Гид: Нино Беридзе ← ['нино', 'нино беридзе']
  Гид: Георгий Кикнадзе ← ['георгий кикнадзе', 'кикнадзе']
  Гид: Ашот Мкртчян ← ['ашот', 'ашот мкртчян']
  Гид: Лилит Акопян ← ['акопян', 'лилит акопян']
  Гид: Мерве Йылдыз ← ['мерве', 'мерве йылдыз']
  Гид: Олег Строганов ← ['олег строганов', 'строганов']
  Гид: Ирина Волкова ← ['волкова', 'ирина волкова']
  Гид: Дмитрий Орлов ← ['дмитрий', 'дмитрий орлов']
  Гид:

## 3. Агент с двумя видами поиска

Поиск по тексту дан готовым: BM25 по документам, как на семинаре. Он возвращает три самых похожих документа целиком.


In [10]:
BM25 = BM25Okapi([tokenize(doc["title"] + " " + doc["text"]) for doc in DOCS])

def text_search(query: str, k: int = 3) -> str:
    scores = BM25.get_scores(tokenize(query))
    top = np.argsort(-scores)[:k]
    hits = [DOCS[i] for i in top if scores[i] > 0]
    if not hits:
        return "По тексту ничего не найдено."
    return "\n\n".join(f"[{doc['doc_id']}] {doc['title']}\n{doc['text']}" for doc in hits)

text_tool = StructuredTool.from_function(
    func=text_search, name="text_search", handle_tool_error=True,
    description="Поиск по документам турагентства: карточки отелей, гидов, экскурсий, туров и правила агентства. "
                "Возвращает три самых похожих документа с [doc_id].",
)
print(text_search("языки экскурсии Гастротур по Казани"))


[e_kzn_food] Экскурсия «Гастротур по Казани»
«Гастротур по Казани» — авторская экскурсия в Казани: чак-чак, эчпочмак и губадия. Длительность 3 ч, стоимость 2900 рублей с человека.

Встреча у входа на Колхозный рынок. Ведёт экскурсию Алия.

[t_volga] Тур «Казанские выходные»
«Казанские выходные» — тур на 3 дня, маршрут: Казань. Стоимость 32000 рублей на человека.

Проживание: «Кремлёвский двор». В программу входят экскурсии «Гастротур по Казани»; «Кремль и Кул-Шариф». Остальное время свободное.

[g_safin] Гид Марат Сафин
Марат Сафин работает с нами в Казани уже 6 лет. Языки: русский, немецкий.

Отзывы туристов о гиде можно прочитать в личном кабинете.


### 3.1. Инструмент поиска по графу

Напишите `make_graph_tool(kg) -> BaseTool` — функцию, которая возвращает инструмент LangChain с именем `graph_search`.

Способ — любой из обычных для LangChain: `StructuredTool.from_function` со схемой аргументов на Pydantic, `@tool` или класс-наследник `BaseTool`.

Требования:

* аргументы:
  * `entities` — список названий сущностей, как они звучат в вопросе (обязательный);
  * `depth` — глубина от 1 до 3; можно добавить и другие, например фильтр связей;
* инструмент находит вершины через `kg.find`, раскрывает подграф через `kg.expand` и возвращает `kg.evidence`; про сущности, которых в графе нет, прямо говорит модели;
* описание объясняет модели, какие вопросы решаются графом, а какие — поиском по тексту;
* неверные аргументы не роняют агента: модель получает текст ошибки.


In [11]:
def make_graph_tool(kg: KnowledgeGraph) -> BaseTool:
    relation_names = list(SCHEMA["relations"]) if "SCHEMA" in globals() else []

    class GraphSearchArgs(BaseModel):
        entities: list[str] = Field(
            description="Названия сущностей так, как они звучат в вопросе: экскурсии, туры, отели, гиды, города, языки. "
                        "Например [\"Гастротур по Казани\"] или [\"Казанские выходные\"]. Каждая сущность — отдельный элемент.")
        depth: int = Field(default=2, description="Сколько шагов по связям пройти от найденных сущностей: от 1 до 3. "
                                                  "1 — прямые связи, 2 — экскурсия → гид → языки, 3 — тур → экскурсии → гиды → языки.")
        relations: Optional[list[str]] = Field(
            default=None,
            description="Необязательный фильтр: по каким связям идти. Допустимые: " + ", ".join(relation_names) + ". "
                        "Если не уверены — не указывайте.")

    def graph_search(entities: list[str], depth: int = 2, relations: Optional[list[str]] = None) -> str:
        if isinstance(entities, str):
            entities = [entities]
        entities = [e.strip() for e in (entities or []) if isinstance(e, str) and e.strip()]
        if not entities:
            raise ToolException("Передайте в entities хотя бы одно название сущности, например [\"Гастротур по Казани\"].")
        notes = []
        try:
            depth = int(depth)
        except (TypeError, ValueError):
            depth = 2
        if not 1 <= depth <= 3:
            notes.append(f"Глубина {depth} вне диапазона 1–3, использую {min(max(depth, 1), 3)}.")
            depth = min(max(depth, 1), 3)
        if relations:
            known = [r for r in relations if r in relation_names] if relation_names else list(relations)
            unknown = [r for r in relations if r not in known]
            if unknown:
                notes.append(f"Неизвестные связи проигнорированы: {', '.join(unknown)}. Допустимые: {', '.join(relation_names)}.")
            relations = known or None

        found, pairs, missing = [], [], []
        for name in entities:
            nodes = kg.find(name, limit=1)
            if not nodes:
                missing.append(name)
                continue
            # Совпадение должно покрывать бóльшую часть запроса: «Ночной Стамбул на яхте» ≠ город «Стамбул»
            q_tokens = kg._tokens(name)
            a_tokens = {t for alias in kg.graph.nodes[nodes[0]]["aliases"] for t in alias.split()}
            covered = sum(any(kg._token_match(q, a) or kg._token_match(kg._canon_language(q), a) for a in a_tokens)
                          for q in q_tokens)
            if q_tokens and covered / len(q_tokens) < 0.5:
                node = kg.graph.nodes[nodes[0]]
                missing.append(f"{name} (похожа только вершина «{node['name']}» ({node['type']}), это другая сущность)")
                continue
            pairs.append((name, nodes[0]))
            if nodes[0] not in found:
                found.append(nodes[0])

        parts = notes[:]
        if not found:
            parts.append(f"В графе не найдено ни одной из сущностей: {', '.join(entities)}. "
                         "Проверьте написание или поищите через text_search; если и там нет — значит, в базе этого нет.")
            return "\n".join(parts)
        parts.append("Найдено в графе: " + "; ".join(
            f"«{name}» → {kg.graph.nodes[n]['name']} ({kg.graph.nodes[n]['type']})" for name, n in pairs))
        if missing:
            parts.append(f"Не найдено в графе: {', '.join(missing)}.")
        edges = kg.expand(found, depth=depth, relations=relations, max_edges=40)
        parts.append(f"Связи (глубина {depth}, {len(edges)} шт.; формат: источник —связь→ цель [doc_id] «цитата»):")
        parts.append(kg.evidence(edges))
        return "\n".join(parts)

    return StructuredTool.from_function(
        func=graph_search,
        name="graph_search",
        args_schema=GraphSearchArgs,
        handle_tool_error=True,
        handle_validation_error=lambda e: f"Ошибка в аргументах graph_search: {e}. "
                                          "Нужно: entities — список строк, depth — целое от 1 до 3.",
        description=(
            "Поиск по графу знаний турагентства: сущности (города, отели, экскурсии, гиды, языки, туры) и связи между ними "
            "(находится_в, работает_в, ведёт, говорит_на, стартует_от, включает, проживание_в), у каждой связи — [doc_id] и цитата. "
            "Используй для вопросов, где ответ складывается из нескольких связей из разных документов: кто ведёт экскурсию и на каких "
            "языках он говорит; какие экскурсии входят в тур и кто их гиды; от какого отеля стартует экскурсия; в каком отеле живут "
            "участники тура; какие гиды работают в городе. Гиды в карточках экскурсий названы коротко (только имя или фамилия) — граф "
            "уже склеил их с полными карточками. НЕ используй для цен, длительности, описаний и правил агентства (возврат, дети, группы, "
            "доплаты) — этих атрибутов в графе нет, для них нужен text_search."
        ),
    )

graph_tool = make_graph_tool(kg)
print(graph_tool.invoke({"entities": ["Гастротур по Казани"], "depth": 2}))


Найдено в графе: «Гастротур по Казани» → Гастротур по Казани (Экскурсия)
Связи (глубина 2, 12 шт.; формат: источник —связь→ цель [doc_id] «цитата»):
Алия Гарипова (Гид) —ведёт→ Гастротур по Казани (Экскурсия) [e_kzn_food] «Ведёт экскурсию Алия»
Казанские выходные (Тур) —включает→ Гастротур по Казани (Экскурсия) [t_volga] «в программу входят экскурсии «Гастротур по Казани»; «Кремль и Кул-Шариф».»
Гастротур по Казани (Экскурсия) —находится_в→ Казань (Город) [e_kzn_food] «авторская экскурсия в Казани»
Гастротур по Казани (Экскурсия) —находится_в→ Казань (Город) [t_volga] «маршрут: Казань.»
Алия Гарипова (Гид) —ведёт→ Старо-Татарская слобода (Экскурсия) [e_kzn_sloboda] «Гид — Алия»
Алия Гарипова (Гид) —говорит_на→ английский (Язык) [g_garipova] «ведёт экскурсии на русском, татарском и английском языках»
Алия Гарипова (Гид) —говорит_на→ русский (Язык) [g_garipova] «ведёт экскурсии на русском, татарском и английском языках»
Алия Гарипова (Гид) —говорит_на→ татарский (Язык) [g_garipova] «ведё

### 3.2. Промпт и агент

1. `AGENT_PROMPT`: когда искать по тексту, когда по графу, когда сочетать; факты — только из результатов инструментов; после каждого факта — `[doc_id]`; если ничего не нашлось — сказать, что в базе этого нет.
2. `build_agent(llm, tools)` — агент на `create_agent` с этим промптом.
3. `ask(agent, question, max_steps=8) -> dict` — вопрос с пустой историей, не больше `max_steps` обращений к модели. Возвращает `answer`, `messages`, `tool_calls` (список пар (имя, аргументы) по порядку), `cited_doc_ids` (`extract_citations`), `llm_calls` (число ответов модели). При превышении лимита — понятный ответ без исключения.


In [12]:
AGENT_PROMPT = """Ты — ассистент турагентства «Попутный ветер». Отвечай на вопросы туристов только по базе агентства.
У тебя два инструмента поиска.

text_search(query) — поиск по тексту документов (BM25), возвращает три самых похожих документа целиком с [doc_id].
  Используй, когда ответ лежит в одном документе: цены, длительность, стоимость ночи, описание отеля или экскурсии,
  правила агентства (отмена и возврат, дети, размер групп, доплаты за язык). Формулируй запрос ключевыми словами
  с названием объекта, например «Кабан цена ночь» или «отмена тура возврат 10 дней».

graph_search(entities, depth, relations) — поиск по графу знаний: сущности (город, отель, экскурсия, гид, язык, тур)
  и связи (находится_в, работает_в, ведёт, говорит_на, стартует_от, включает, проживание_в) с [doc_id] и цитатой.
  Используй, когда ответ складывается из нескольких связей из разных документов:
  экскурсия → гид → языки (depth=2); тур → экскурсии → гиды → языки (depth=3); тур → отель ← экскурсия (depth=2–3);
  отель ← экскурсия → гид (depth=2); город ← гиды → языки (depth=2). В entities передавай названия как в вопросе.
  Гиды в карточках экскурсий названы коротко (только имя или фамилия) — граф уже связал их с полной карточкой гида.

Как действовать:
1. Сначала реши, сколько шагов нужно для ответа. Один документ — text_search. Цепочка связей — graph_search с подходящей depth.
2. Сочетай инструменты: если граф нашёл объект, а нужен его атрибут (цена отеля, длительность), сделай text_search по найденному
   названию. Если граф не нашёл сущность или связь выглядит неполной — проверь через text_search, и наоборот.
3. Не повторяй один и тот же вызов с теми же аргументами. Обычно хватает 1–3 вызовов, не больше 5.
4. В вопросах «все ли», «есть ли общий», «сможет ли» явно перечисли все объекты (каждую экскурсию, каждого гида и его языки)
   и сравни; ответ на вопрос да/нет начинай со слова «Да» или «Нет».

Правила ответа:
- Используй только факты из результатов инструментов. Ничего не додумывай и не бери из общих знаний.
- После каждого факта ставь ссылку на документ-источник в квадратных скобках: [h_kaban]; несколько — [e_kzn_food, g_garipova].
  doc_id копируй точно из результатов инструментов. Для фактов из цепочки ссылайся на все документы, из которых взяты звенья.
- Людей называй полным именем, если оно есть в результатах (например, «Алия Гарипова», а не «Алия»).
- Если после поиска обоими инструментами ответа нет, прямо напиши: «В базе нет информации о …» — и в этом случае
  не ставь никаких ссылок [doc_id] и не придумывай похожие варианты.
- Отвечай по-русски, кратко и по существу."""

def build_agent(llm, tools: list) -> Any:
    tools = list(tools)
    names = sorted(t.name for t in tools)
    unavailable = [n for n in ("text_search", "graph_search") if n not in names]
    prompt = AGENT_PROMPT
    if unavailable:
        # Промпт описывает оба инструмента; если какого-то нет, модель пытается его вызвать и возвращает пустой ответ.
        prompt += (f"\n\nВАЖНО: в этом запуске доступны только инструменты: {', '.join(names)}. "
                   f"Инструмент(ы) {', '.join(unavailable)} недоступны — не вызывай их. "
                   "Для вопросов-цепочек делай несколько поисков подряд: сначала найди карточку экскурсии или тура, "
                   "затем отдельным запросом — карточку найденного гида или отеля.")
    return create_agent(model=llm, tools=tools, system_prompt=prompt)

def ask(agent, question: str, max_steps: int = 8) -> dict:
    # В графе create_agent каждое обращение к модели и каждый запуск инструментов — отдельный шаг LangGraph:
    # модель → инструменты → модель → … Поэтому max_steps обращений к модели = 2 * max_steps - 1 шагов.
    messages: list = [HumanMessage(content=question)]
    stopped, llm_calls = False, 0
    for attempt in range(2):  # вторая попытка — только если модель вернула пустой ответ
        steps_left = max_steps - llm_calls
        if steps_left <= 0:
            stopped = True
            break
        config = {"recursion_limit": max(1, 2 * steps_left - 1)}
        try:
            for state in agent.stream({"messages": messages}, config=config, stream_mode="values"):
                if isinstance(state, dict) and "messages" in state:
                    messages = state["messages"]
        except GraphRecursionError:
            stopped = True
        llm_calls = sum(isinstance(m, AIMessage) for m in messages)
        final = messages[-1] if messages and isinstance(messages[-1], AIMessage) else None
        empty = final is not None and not final.tool_calls and not final.text.strip()
        if stopped or not empty:
            break
        messages = list(messages) + [HumanMessage(content="Ответ пустой. Вызови инструмент поиска или дай итоговый ответ "
                                                          "по уже найденному, с [doc_id] после каждого факта.")]

    ai_messages = [m for m in messages if isinstance(m, AIMessage)]
    tool_calls = [(call["name"], dict(call.get("args") or {})) for m in ai_messages for call in (m.tool_calls or [])]
    final = ai_messages[-1] if ai_messages else None
    answer = final.text.strip() if final is not None and not final.tool_calls else ""
    if stopped or not answer:
        partial = next((m.text.strip() for m in reversed(ai_messages) if m.text and m.text.strip()), "")
        reason = (f"агент превысил лимит в {max_steps} обращений к модели" if stopped
                  else "модель вернула пустой ответ")
        answer = f"Не удалось получить ответ: {reason}." + (f" Последний промежуточный ответ: {partial}" if partial else "")
    return {
        "answer": answer,
        "messages": messages,
        "tool_calls": tool_calls,
        "cited_doc_ids": extract_citations(answer),
        "llm_calls": len(ai_messages),
    }



In [13]:
graph_agent = build_agent(llm, [text_tool, graph_tool])
for qid in ["q03", "q07", "q15", "q21"]:
    item = next(i for i in EVAL if i["qid"] == qid)
    result = ask(graph_agent, item["question"])
    print(f"=== {qid} ({item['hops']} шаг.): {item['question']}")
    for message in result["messages"]:
        if isinstance(message, AIMessage):
            for call in message.tool_calls:
                print(f"  Вызов: {call['name']}({call['args']})")
        elif isinstance(message, ToolMessage):
            print(f"    Результат: {message.text[:200].replace(chr(10), ' / ')}")
    print(f"  Ответ: {result['answer']}\n")


=== q03 (1 шаг.): Сколько стоит ночь в отеле «Кабан» в Казани?
  Вызов: text_search({'query': 'Кабан цена ночь'})
    Результат: [h_kaban] Гостиница «Кабан» / В Казани мы часто селим туристов в «Кабан»: на берегу озера Кабан, рядом Старо-Татарская слобода. Цена — от 5100 рублей за ночь. /  / Заезд с 14:00, выезд до 12:00. Бронировани
  Ответ: Ночь в отеле «Кабан» в Казани стоит от 5100 рублей [h_kaban].

=== q07 (2 шаг.): На каких языках можно провести экскурсию «Гастротур по Казани»?
  Вызов: graph_search({'depth': 2, 'entities': ['Гастротур по Казани'], 'relations': ['ведёт', 'говорит_на']})
    Результат: Найдено в графе: «Гастротур по Казани» → Гастротур по Казани (Экскурсия) / Связи (глубина 2, 5 шт.; формат: источник —связь→ цель [doc_id] «цитата»): / Алия Гарипова (Гид) —ведёт→ Гастротур по Казани (Экс
  Ответ: Экскурсия «Гастротур по Казани» проводится на русском, татарском и английском языках [g_garipova].

=== q15 (3 шаг.): На каких языках говорят гиды экскурсий, входящих в т

### 3.3. Что делает агент

По выводу выше (3–5 предложений): какой поиск агент выбрал для вопроса, на который отвечает один документ, и для вопросов, где ответ складывается из нескольких связей (экскурсия → гид → языки); хватило ли ему графа или он дочитывал документы; что он ответил на вопрос, ответа на который в базе нет.

**Analysis / Разбор:**  На вопрос с ответом в одном документе (q03) агент сразу выбрал text_search с запросом «Кабан цена ночь» и ответил одним вызовом: 5100 рублей [h_kaban]. Для цепочки экскурсия → гид → языки (q07) он вызвал graph_search с depth=2 и фильтром связей «ведёт», «говорит_на», получил 5 рёбер и ответил по графу, не дочитывая документы (правда, сослался только на [g_garipova]). В трёхшаговом q15 агент тоже ограничился одним graph_search (depth=3, 7 рёбер), хотя языков большинства гидов в графе не было, и придумал ответ «все гиды говорят на русском и английском» с несуществующей цитатой «Гид — Георгий Кикнадзе»; здесь надо было дочитать карточки гидов через text_search. На вопрос без ответа в базе (q21) он поискал в графе «гид», ничего не нашёл и честно ответил, что информации нет, без ссылок на документы.


Обычный RAG для сравнения дан готовым: один текстовый поиск по вопросу и один запрос к модели.


In [14]:
CLASSIC_PROMPT = (
    "Ответь на вопрос только по фрагментам базы турагентства. После каждого факта ставь [doc_id]. "
    "Если ответа во фрагментах нет — так и скажи.\n\nФрагменты:\n{context}\n\nВопрос: {question}"
)

def classic_rag(question: str) -> dict:
    context = text_search(question)
    reply = llm.invoke(CLASSIC_PROMPT.format(context=context, question=question)).text
    return {"answer": reply, "messages": [], "tool_calls": [("text_search", {"query": question})],
            "cited_doc_ids": extract_citations(reply), "llm_calls": 1}


## 4. Сравнение

Сравним три системы на 10 вопросах из 24 (`COMPARE_QIDS`, по несколько на каждое число шагов), чтобы не тратить много запросов: обычный RAG, агента только с текстовым поиском и агента с текстом и графом. Метрики по вопросу считает готовая `score_answer`:

* `facts_recall` — доля ключевых фактов, найденных в ответе;
* `citation_ok` — процитирован ли хотя бы один нужный документ (для вопросов без ответа в базе — `None`);
* `honest` — для вопросов без ответа в базе: признал ли агент, что информации нет, и не сослался ли на источники; для остальных — `None`;
* `graph_calls`, `text_calls`, `llm_calls` — сколько было вызовов графа, текста и модели.


In [15]:
METRICS = ["facts_recall", "citation_ok", "honest", "graph_calls", "text_calls", "llm_calls"]
HOP_NAMES = {0: "нет в базе", 1: "1 шаг", 2: "2 шага", 3: "3 шага"}

def score_answer(item: dict, result: dict) -> dict:
    answer = (result["answer"] or "").lower().replace("ё", "е")
    facts = item["key_facts"]
    recall = (sum(any(v.lower().replace("ё", "е") in answer for v in variants) for variants in facts) / len(facts)) if facts else None
    cited = set(result["cited_doc_ids"])
    return {
        "facts_recall": recall,
        "citation_ok": bool(cited & set(item["gold_doc_ids"])) if item["gold_doc_ids"] else None,
        "honest": (has_no_info(result["answer"]) and not cited) if item["hops"] == 0 else None,
        "graph_calls": sum(name == "graph_search" for name, _ in result["tool_calls"]),
        "text_calls": sum(name == "text_search" for name, _ in result["tool_calls"]),
        "llm_calls": result["llm_calls"],
    }

def run_eval(system_fn: Callable, items: list[dict]) -> list[dict]:
    results = parallel_map(lambda item: system_fn(item["question"]), items, "вопросы")
    rows = []
    for item, result in zip(items, results):
        rows.append({"qid": item["qid"], "hops": item["hops"], **score_answer(item, result),
                     "answer": result["answer"], "tool_calls": result["tool_calls"]})
    return rows

def print_rows(title: str, rows: list[dict]) -> None:
    print(f"\n{title}")
    mark = lambda v: "—" if v is None else ("да" if v is True else ("нет" if v is False else f"{v:.2f}"))
    for row in rows:
        print(f"{row['qid']} {HOP_NAMES[row['hops']]:<10} факты {mark(row['facts_recall']):<5} цитата {mark(row['citation_ok']):<4} "
              f"честно {mark(row['honest']):<4} граф {row['graph_calls']} текст {row['text_calls']} | {row['answer'][:70]!r}")


Сводку по группам вопросов считает функция `summarize`: для каждой группы по числу шагов и для всех вопросов вместе — число вопросов и среднее по каждой метрике (`None` в среднем не участвует).


In [16]:
def summarize(rows: list[dict]) -> dict:
    groups: dict[str, list[dict]] = defaultdict(list)
    for row in rows:
        groups[HOP_NAMES[row["hops"]]].append(row)
        groups["ВСЕГО"].append(row)
    table = {}
    for name, items in groups.items():
        values = {"n": len(items)}
        for metric in METRICS:
            present = [float(item[metric]) for item in items if item[metric] is not None]
            values[metric] = round(sum(present) / len(present), 3) if present else None
        table[name] = values
    return table


In [17]:
COMPARE_QIDS = ["q03", "q05", "q07", "q10", "q13", "q15", "q16", "q17", "q19", "q21"]
COMPARE = [item for item in EVAL if item["qid"] in COMPARE_QIDS]

text_agent = build_agent(llm, [text_tool])
classic_rows = run_eval(classic_rag, COMPARE)
text_rows = run_eval(lambda q: ask(text_agent, q), COMPARE)
graph_rows = run_eval(lambda q: ask(graph_agent, q), COMPARE)

print_rows("Обычный RAG", classic_rows)
print_rows("Агент: только текст", text_rows)
print_rows("Агент: текст + граф", graph_rows)

for title, rows in [("Обычный RAG", classic_rows), ("Агент: только текст", text_rows), ("Агент: текст + граф", graph_rows)]:
    table = summarize(rows)
    print(f"\n{title}")
    print(f"{'группа':<12}{'n':>3}" + "".join(f"{m:>14}" for m in METRICS))
    for name in sorted(table, key=lambda n: (n == "ВСЕГО", n)):
        values = table[name]
        print(f"{name:<12}{values['n']:>3}" + "".join(f"{'—' if values[m] is None else round(values[m], 2):>14}" for m in METRICS))


вопросы: готово 2/10
вопросы: готово 4/10
вопросы: готово 6/10
вопросы: готово 8/10
вопросы: готово 10/10
вопросы: готово 2/10
вопросы: готово 4/10
вопросы: готово 6/10
вопросы: готово 8/10
вопросы: готово 10/10
вопросы: готово 2/10
вопросы: готово 4/10
вопросы: готово 6/10
вопросы: готово 8/10
вопросы: готово 10/10

Обычный RAG
q03 1 шаг      факты 1.00  цитата да   честно —    граф 0 текст 1 | 'Отель «Кабан» в Казани стоит от 5100 рублей за ночь [h_kaban].'
q05 1 шаг      факты 1.00  цитата да   честно —    граф 0 текст 1 | 'Вернут 50% стоимости тура, если отменить тур за 10 дней до выезда [pol'
q07 2 шага     факты 1.00  цитата да   честно —    граф 0 текст 1 | 'На русском, татарском и английском языках [g_garipova].'
q10 2 шага     факты 1.00  цитата да   честно —    граф 0 текст 1 | 'Гид экскурсии «Кремль и Кул-Шариф» не указан как говорящий по-немецки.'
q13 2 шага     факты 0.00  цитата да   честно —    граф 0 текст 1 | 'Экскурсия «Гастротур по Казани» стартует от отеля, где живу

### 4.1. Выводы

По сводкам и построчным выводам, ссылаясь на цифры и номера вопросов:

1. На каких вопросах граф помог и почему: что агент с графом сделал иначе, чем агент только с текстом?
2. Где граф не помог или помешал? Есть ли вопросы, где агент с текстом справился лучше? Чем это объясняется — ошибками вашего графа, выбором инструмента или чем-то ещё?
3. Сколько стоит каждая система (вызовы инструментов и модели) и что бы вы выбрали для такой базы?

**Analysis / Разбор:**Лучший результат у агента с графом: facts_recall 0.8 против 0.62 у обычного RAG и 0.56 у агента только с текстом, citation_ok 1.0, и только он получил honest = 1 на q21. Сильнее всего граф помог на двухшаговых вопросах (1.0 против 0.67 у RAG и 0.28 у текстового агента): в q07 текстовый агент прочитал только карточку экскурсии и назвал один русский язык (0.33), а агент с графом за один graph_search прошёл экскурсия → Алия Гарипова → языки; в q13 и RAG, и текстовый агент ошибочно ответили, что от «Кремлёвского двора» стартует «Гастротур по Казани» (0.00), а по графу нашлась правильная экскурсия «Кремль и Кул-Шариф» (1.00). Так получается, потому что в графе короткие имена из карточек экскурсий («Сафин», «Алия», «Петроградка») уже склеены с полными вершинами.
На трёхшаговых вопросах граф не дал преимущества: 0.54, как у текстового агента (у RAG 0.4). В q15 лучше всех оказался RAG (0.75 против 0.50), в q19 все три системы получили 0.00. Здесь виноват мой граф: при извлечении потерялись почти все связи «говорит_на» (4 вместо ~25), а агент доверял неполному подграфу и почти не проверял его текстом (0.3 вызова text_search в среднем). Текстовый агент в q15 и q17 сделал 4 и 3 поиска и достал часть языков из карточек гидов, но тоже ответил не полностью. Кроме того, метрики не идеальны: текстовый агент на q21 честно ответил «в базе нет гида…», но не попал в регулярное выражение has_no_info, а 1.00 у агента с графом в q10 может быть ложным: метрика нашла подстроку «говорит» в ответе «говорит по-русски».
По стоимости RAG самый дешёвый (1 вызов модели на вопрос), агент с графом тратит 2.3 вызова модели и 1.3 вызова инструментов, текстовый агент — 2.5 и 1.5 (на трёхшаговых до 3.25 вызова модели); плюс к графу один раз 47 вызовов на извлечение. Для такой базы я бы выбрал агента с графом, но улучшил бы извлечение и заставил агента проверять неполный подграф текстовым поиском; на простые одношаговые вопросы хватает обычного RAG.



## 5. Разбор ошибок

### 5.1. Три ошибки по уровням

Выберите из прогона агента с графом три вопроса, где что-то пошло не так. Для каждого: что произошло (по вызовам, найденному и ответу) и на каком уровне ошибка — извлечение, слияние сущностей, поиск вершин, раскрытие подграфа, выбор инструмента агентом или генерация ответа, — и как бы вы её исправили: что поменять и на каком уровне (одно-два предложения, код писать не нужно). Ячейка ниже печатает подробности — впишите номера вопросов.

Если явных ошибок меньше трёх, доберите вопросы, где агент ответил верно, но неоптимально: лишние или повторные вызовы, не тот инструмент с первой попытки, подграф из сотни рёбер ради одного факта, нет ссылки на нужный документ. Если и таких нет — сломайте один уровень сами: например, пусть `resolve` всегда возвращает `None` (сущности не сливаются) или `expand` идёт только на один шаг, — пересоберите граф и разберите, какие вопросы сломались и почему.


In [22]:
ERROR_QIDS = ["q15", "q17", "q19"]  # TODO 5.1: три номера вопросов, например ["q16", "q17", "q19"]


In [23]:
for qid in ERROR_QIDS:
    row = next(r for r in graph_rows if r["qid"] == qid)
    item = next(i for i in EVAL if i["qid"] == qid)
    print(f"===== {qid}: {item['question']}")
    print("Нужные документы:", item["gold_doc_ids"], "| ключевые факты:", item["key_facts"])
    print("Вызовы:", row["tool_calls"])
    for name, args in row["tool_calls"]:
        if name == "graph_search":
            print("  Граф нашёл:", [kg.graph.nodes[n]["name"] for e in args.get("entities", []) for n in kg.find(e, limit=1)])
    print("Ответ:", row["answer"], "\n")


===== q15: На каких языках говорят гиды экскурсий, входящих в тур «Тбилиси и Ереван за неделю»?
Нужные документы: ['t_caucasus', 'e_tbs_baths', 'e_tbs_khinkali', 'e_evn_garni', 'g_kiknadze', 'g_beridze', 'g_mkrtchyan'] | ключевые факты: [['грузин'], ['русск'], ['армян'], ['англий']]
Вызовы: [('graph_search', {'depth': 3, 'entities': ['Тбилиси и Ереван за неделю'], 'relations': ['включает', 'ведёт', 'говорит_на']})]
  Граф нашёл: ['Тбилиси и Ереван за неделю']
Ответ: Гиды экскурсий, входящих в тур «Тбилиси и Ереван за неделю», говорят на следующих языках:

- Георгий Кикнадзе (ведёт экскурсию «Бани и Абанотубани») — говорит на русском и английском [e_tbs_baths] «Гид — Георгий Кикнадзе».
- Ашот Мкртчян (ведёт экскурсию «Гарни и Гегард») — говорит на русском и английском [e_evn_garni] «Гид — Ашот».
- Нино Беридзе (ведёт экскурсии «Мастер-класс по хинкали» и «Фуникулёр и Мтацминда») — говорит на русском и английском [e_tbs_khinkali, e_tbs_mtats] «Гид — Нино».

Все гиды говорят на русском и 

Для каждого из трёх вопросов: что произошло, на каком уровне ошибка и как бы вы её исправили.

**Analysis / Разбор:** **q15.** Агент вызвал graph_search по туру с depth=3 и получил всего 7 рёбер: экскурсии и их гиды есть, а связей «говорит_на» у Кикнадзе, Беридзе и Мкртчяна нет — при извлечении они были отброшены, потому что модель не внесла языки в entities. Не проверив карточки гидов текстом, агент придумал «все гиды говорят на русском и английском». Ошибка на уровне извлечения (и затем генерации ответа); исправил бы промпт 1.1 (каждый конец связи — в entities) или автодобавление сущности «Язык» в parse_extraction, а в AGENT_PROMPT запретил бы отвечать про то, чего нет в подграфе.
**q17.** Агент сделал три вызова graph_search и один text_search и ответил, что информации о языках Мерве Йылдыз и Строганова нет. Языков этих гидов в графе нет (ошибка извлечения), а единственный текстовый поиск не достал их карточки — выбор запроса агентом. Нужно исправить извлечение и подсказать агенту искать карточку каждого гида отдельным text_search по имени.
**q19.** Ответ «гиды говорят на английском, татарском и русском» — это языки одной Гариповой; связи «говорит_на» у Сафина (русский, немецкий) были отброшены ещё в 1.2, так что общих языков посчитать было нельзя. Уровень — извлечение, плюс генерация: агент не сравнил языки по каждому гиду и не сказал, что данных по Сафину нет. Исправление — то же в извлечении, а в промпте — требовать перечислить языки каждого гида перед выводом об общем.
